# Notebook 02 — Best-of-N vs Self-Consistency at Fixed Budget

**Chapter**: [Chapter 3 — Sampling and Verification](../chapters/03-sampling-and-verification.md).

**Claim demonstrated**: At a fixed total token budget on a small open model, *best-of-N with a verifier* beats *self-consistency (majority vote)* by a measurable margin, and both beat *single-sample long-CoT*. The crossover budget where BoN > self-consistency depends on the task and the verifier.

**Hardware**: ≥ 16 GB GPU.

**Dependencies**: as in notebook 01.

---

## Setup

We hold *total tokens per problem* constant (`TOTAL_BUDGET`). Three strategies at that budget:
- **Long-CoT**: one sample with `max_new_tokens = TOTAL_BUDGET`.
- **Self-Consistency (K)**: `K = TOTAL_BUDGET / SHORT_BUDGET` samples of length `SHORT_BUDGET`, majority vote.
- **Best-of-N (K)**: same K samples, but reranked by a small reward model.

Reward model: any small open math-tuned verifier (e.g. `Qwen2.5-Math-RM-72B` if you have the compute; we use a smaller proxy by default).

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B-Instruct"
RM_NAME = "Qwen/Qwen2.5-Math-PRM-7B"  # replace with the smallest RM you can fit
TOTAL_BUDGET = 1024
SHORT_BUDGET = 256
K = TOTAL_BUDGET // SHORT_BUDGET  # number of short samples = 4
N_PROBLEMS = 30
SEED = 42

import os, random, json, math, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from collections import Counter
from tqdm.auto import tqdm

random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Load policy.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map=device,
)
model.eval()

In [ ]:
# Load PRM (process reward model). Skip if too large.
rm_tokenizer = AutoTokenizer.from_pretrained(RM_NAME)
rm = AutoModelForSequenceClassification.from_pretrained(
    RM_NAME, torch_dtype=torch.bfloat16, device_map=device, trust_remote_code=True,
)
rm.eval()

In [ ]:
# Data and helpers (re-use from notebook 01).
ds = load_dataset("HuggingFaceH4/MATH-500", split="test").shuffle(seed=SEED).select(range(N_PROBLEMS))
BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")

def extract_answer(text):
    m = BOXED_RE.findall(text)
    return m[-1].strip() if m else None

def is_correct(pred, gold):
    if pred is None: return False
    norm = lambda s: re.sub(r"\s+", "", s).lower()
    if norm(pred) == norm(gold): return True
    try: return abs(float(pred) - float(gold)) < 1e-6
    except: return False

In [ ]:
def sample_completion(prompt, max_new_tokens, temperature=0.8):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=temperature > 0, temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def rm_score(problem, chain_with_answer):
    """Score a (problem, chain) pair with the PRM. Higher = better."""
    text = f"Problem: {problem}\nSolution: {chain_with_answer}"
    inputs = rm_tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(device)
    with torch.inference_mode():
        out = rm(**inputs)
    return float(out.logits.squeeze().item())

In [ ]:
# Run the three strategies on each problem.
n_correct = {"long_cot": 0, "self_consistency": 0, "best_of_n": 0}

for ex in tqdm(ds, desc="problems"):
    problem = ex["problem"]
    gold = ex["answer"]
    prompt = f"Solve the following math problem. Put your final answer in \\boxed{{}}.\n\n{problem}\n"

    # Long-CoT
    long_gen = sample_completion(prompt, TOTAL_BUDGET, temperature=0.0)
    n_correct["long_cot"] += int(is_correct(extract_answer(long_gen), gold))

    # K short samples
    chains = [sample_completion(prompt, SHORT_BUDGET, temperature=0.8) for _ in range(K)]
    answers = [extract_answer(c) for c in chains]

    # Self-consistency: majority vote among non-None.
    valid = [a for a in answers if a is not None]
    sc_answer = Counter(valid).most_common(1)[0][0] if valid else None
    n_correct["self_consistency"] += int(is_correct(sc_answer, gold))

    # Best-of-N: rerank by RM.
    scores = [rm_score(problem, c) for c in chains]
    best_idx = max(range(K), key=lambda i: scores[i])
    n_correct["best_of_n"] += int(is_correct(answers[best_idx], gold))

for k, v in n_correct.items():
    print(f"{k}: {v}/{N_PROBLEMS} = {v/N_PROBLEMS:.3f}")

## Interpretation

Expected ordering on MATH-500 with TOTAL_BUDGET=1024, K=4:

```
best_of_n  >  self_consistency  >  long_cot
```

With small N=30 and K=4, the gap between `best_of_n` and `self_consistency` will be a few accuracy points (1 – 5%). At larger K (16, 32), the gap widens. The gap between `self_consistency` and `long_cot` is task-dependent; on easy problems where 256 tokens is sufficient, self-consistency dominates; on hard problems where 1024 tokens are needed for the chain to fit, long-cot can match or beat.

**Caveats:**
- N=30 is a *demonstration*, not a benchmark. Real numbers need N ≥ 500.
- The RM choice matters a lot. A weak RM makes BoN worse than self-consistency. Try several.
- The temperature for short samples should be reported and held constant across self-consistency and best-of-N for a fair compare.

## Self-refine baseline (optional)

Replace one of the short samples with a self-refine pass: feed the model its own first attempt and ask it to critique. Per Huang et al. (2024), expect this to *underperform* the original sample on math without an external check. The cell below sketches the experiment.

In [ ]:
def self_refine(problem, first_attempt, max_new_tokens=SHORT_BUDGET):
    critique_prompt = (
        f"Problem: {problem}\n"
        f"First attempt: {first_attempt}\n"
        f"Identify any errors and produce a corrected solution. Put your final answer in \\boxed{{}}.\n"
    )
    return sample_completion(critique_prompt, max_new_tokens, temperature=0.0)

# Quick demo on the first problem.
ex = ds[0]
prompt = f"Solve the following math problem. Put your final answer in \\boxed{{}}.\n\n{ex['problem']}\n"
first = sample_completion(prompt, SHORT_BUDGET, temperature=0.0)
refined = self_refine(ex["problem"], first)
print("First:", extract_answer(first))
print("Refined:", extract_answer(refined))
print("Gold:", ex["answer"])